# Cell 1: Imports and path setup


In [1]:
import sys
import os
import time
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

# Make ``src`` importable when running from the ``notebooks/`` directory.
try:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent
except NameError:
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.retriever import Retriever, create_retriever_callable
from src.agents.hyde import HyDEAgent


c:\Users\hanaz\Documents\GitHub\Multi-Agent-Ensemble-for-Search-Through-Reinforcement-Optimization-MAESTRO-\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Cell 2: Configuration


In [2]:
load_dotenv(PROJECT_ROOT / ".env")

class Config:
    # Data paths
    QUERIES_PATH = PROJECT_ROOT / "notebooks" / "queries" / "topics.ms-marco-dev2.tsv"
    QRELS_PATH = PROJECT_ROOT / "notebooks" / "qrels" / "qrels.ms-marco-dev2.tsv"

    # Evaluation scope
    NUM_QUERIES = 20          # Set to an int (e.g. 50) to evaluate a subset.
    NDCG_K = 10               # nDCG@10.
    RECALL_K = 100            # Recall@100 for quick tests.

    # Agent hyperparameters
    TOP_K = 50                # BM25 retrieval and RRF fusion window.
    RRF_K = 60                # RRF smoothing parameter.

    # Output
    OUTPUT_DIR = PROJECT_ROOT / "outputs"
    OUTPUT_CSV = OUTPUT_DIR / "hyde_isolation_results.csv"


cfg = Config()
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Shared embedding model (required by AgentBase, not used by HyDE itself).
EMBED_MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6738.10it/s]


# Cell 3: Data loading helpers


In [3]:
def load_qrels(qrels_path: Path) -> Dict[str, Dict[str, int]]:
    """Load qrels as ``{query_id: {doc_id: relevance_grade}}``."""
    qrels = defaultdict(dict)
    if not qrels_path.exists():
        raise FileNotFoundError(f"Qrels file not found: {qrels_path}")

    with open(qrels_path, "r", encoding="utf-8") as f:
        next(f, None)  # Skip header.
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) < 4:
                continue
            query_id, doc_id, grade_str = parts[0].strip(), parts[2].strip(), parts[3].strip()
            try:
                grade = int(grade_str)
            except ValueError:
                continue
            qrels[query_id][doc_id] = grade
    return dict(qrels)


def load_queries(queries_path: Path, num_queries: int = None) -> List[Tuple[str, str]]:
    """Load queries as ``[(query_id, query_text), ...]``."""
    queries = []
    if not queries_path.exists():
        raise FileNotFoundError(f"Queries file not found: {queries_path}")

    with open(queries_path, "r", encoding="utf-8") as f:
        next(f, None)  # Skip header.
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) >= 2:
                query_id, query_text = parts[0].strip(), parts[1].strip()
            else:
                query_id, query_text = str(len(queries)), parts[0].strip()
            queries.append((query_id, query_text))
            if num_queries is not None and len(queries) >= num_queries:
                break
    return queries


# Cell 4: Metric helpers (mirror Simulation.compute_ndcg / compute_recall)


In [4]:
def _dcg(relevances: np.ndarray, k: int) -> float:
    relevances = np.asarray(relevances, dtype=float)[:k]
    if relevances.size == 0:
        return 0.0
    positions = np.arange(2, relevances.size + 2)
    return float(np.sum(relevances / np.log2(positions)))


def normalize_doc_id(doc_id: str) -> str:
    """Strip segment suffix (e.g. 'doc#1' -> 'doc') to match qrels format."""
    return doc_id.split("#", 1)[0] if "#" in doc_id else doc_id


def deduplicate_doc_ids(doc_ids: List[str]) -> List[str]:
    """Normalize then deduplicate doc IDs, matching Simulation.deduplicate_doc_ids."""
    deduped = []
    seen = set()
    for doc_id in doc_ids:
        normalized = normalize_doc_id(doc_id)
        if normalized not in seen:
            deduped.append(normalized)
            seen.add(normalized)
    return deduped


def compute_ndcg(ranked_doc_ids: List[str], qrels: Dict[str, int], k: int = 10) -> float:
    ranked_docs = deduplicate_doc_ids(ranked_doc_ids)[:k]
    gains = [qrels.get(doc_id, 0) for doc_id in ranked_docs]
    ideal = sorted((rel for rel in qrels.values() if rel > 0), reverse=True)[:k]
    dcg = _dcg(np.array(gains, dtype=float), k)
    idcg = _dcg(np.array(ideal, dtype=float), k)
    return dcg / idcg if idcg > 0 else 0.0


def compute_recall(ranked_doc_ids: List[str], qrels: Dict[str, int], k: int = 100) -> float:
    ranked_docs = deduplicate_doc_ids(ranked_doc_ids)[:k]
    relevant = {d for d, r in qrels.items() if r > 0}
    if not relevant:
        return 0.0
    return len(set(ranked_docs) & relevant) / len(relevant)


# Cell 5: Initialize retriever and HyDE agent (API-based LLM)


In [5]:
retriever_instance = Retriever(
    endpoint=os.getenv("RETRIEVAL_ENDPOINT"),
    username=os.getenv("MY_USERNAME"),
    password=os.getenv("MY_PASSWORD"),
    index_field="segment",
    top_k=cfg.TOP_K,
)
retriever_func = create_retriever_callable(retriever_instance)

# HyDEAgent now calls an LLM API (LLMAPI_KEY / BASE_URL_HPC / MODEL_NAME_HPC from .env).
# No local model weights are downloaded.
hyde_agent = HyDEAgent(
    embed_model=EMBED_MODEL,
    top_k=cfg.TOP_K,
    rrf_k=cfg.RRF_K,
)


# Cell 6: Load data


In [6]:
queries = load_queries(cfg.QUERIES_PATH, num_queries=cfg.NUM_QUERIES)
qrels = load_qrels(cfg.QRELS_PATH)

print(f"Loaded {len(queries)} queries.")
print(f"Loaded qrels for {len(qrels)} queries.")


Loaded 20 queries.
Loaded qrels for 5000 queries.


# Cell 7: Run isolated evaluation and cache rankings


In [7]:
records = []

for query_id, query_text in queries:
    # Baseline BM25
    bm25_start = time.time()
    bm25_doc_ids, bm25_scores, _ = retriever_func(query_text, cfg.TOP_K)
    bm25_elapsed = time.time() - bm25_start

    # HyDE pseudo-document generation + dual retrieval + RRF
    effects = hyde_agent.compute_effects({
        "query_text": query_text,
        "retriever": retriever_func,
        "top_k": cfg.TOP_K,
    })

    hyde_doc_ids = effects["new_doc_ids"]
    hyde_elapsed = effects["elapsed_time"]
    pseudo_document = effects["new_query_text"]

    qrels_for_query = qrels.get(query_id, {})

    # Metrics
    bm25_ndcg = compute_ndcg(bm25_doc_ids, qrels_for_query, k=cfg.NDCG_K)
    hyde_ndcg = compute_ndcg(hyde_doc_ids, qrels_for_query, k=cfg.NDCG_K)
    bm25_recall = compute_recall(bm25_doc_ids, qrels_for_query, k=cfg.RECALL_K)
    hyde_recall = compute_recall(hyde_doc_ids, qrels_for_query, k=cfg.RECALL_K)

    records.append({
        "query_id": query_id,
        "query_text": query_text,
        "pseudo_document": pseudo_document,
        "bm25_doc_ids": ";".join(bm25_doc_ids),
        "hyde_doc_ids": ";".join(hyde_doc_ids),
        "bm25_ndcg": bm25_ndcg,
        "hyde_ndcg": hyde_ndcg,
        "ndcg_gain": hyde_ndcg - bm25_ndcg,
        "bm25_recall": bm25_recall,
        "hyde_recall": hyde_recall,
        "recall_gain": hyde_recall - bm25_recall,
        "bm25_latency_ms": bm25_elapsed * 1000,
        "hyde_latency_ms": hyde_elapsed * 1000,
        "pseudo_extra_tokens": len(pseudo_document.split()) - len(query_text.split()),
    })

# Save enriched cache
df = pd.DataFrame(records)
df.to_csv(cfg.OUTPUT_CSV, index=False)
print(f"Saved rankings + metrics to: {cfg.OUTPUT_CSV}")


c:\Users\hanaz\Documents\GitHub\Multi-Agent-Ensemble-for-Search-Through-Reinforcement-Optimization-MAESTRO-\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'opensearch.pads.fim.uni-passau.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\hanaz\Documents\GitHub\Multi-Agent-Ensemble-for-Search-Through-Reinforcement-Optimization-MAESTRO-\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'opensearch.pads.fim.uni-passau.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\hanaz\Documents\GitHub\Multi-Agent-Ensemble-for-Search-Through-Reinforcement-Optimization-MAESTRO-\.venv\Lib\site-packages\urllib3\connectionpool.py:

Saved rankings + metrics to: c:\Users\hanaz\Documents\GitHub\Multi-Agent-Ensemble-for-Search-Through-Reinforcement-Optimization-MAESTRO-\outputs\hyde_isolation_results.csv


# Cell 8-alt: Recompute metrics from cached rankings + NEW qrels


In [ ]:
# Load the cached results (produced by Cell 7 above)
df_cached = pd.read_csv(cfg.OUTPUT_CSV)

# Load the CORRECT qrels (change path here if needed)
CORRECT_QRELS_PATH = cfg.QRELS_PATH  # or override: Path("data/qrels/correct.qrels.tsv")
correct_qrels = load_qrels(CORRECT_QRELS_PATH)

records = []

for _, row in df_cached.iterrows():
    query_id = row["query_id"]
    qrels_for_query = correct_qrels.get(query_id, {})

    bm25_doc_ids = row["bm25_doc_ids"].split(";") if pd.notna(row["bm25_doc_ids"]) else []
    hyde_doc_ids = row["hyde_doc_ids"].split(";") if pd.notna(row["hyde_doc_ids"]) else []

    bm25_ndcg = compute_ndcg(bm25_doc_ids, qrels_for_query, k=cfg.NDCG_K)
    hyde_ndcg = compute_ndcg(hyde_doc_ids, qrels_for_query, k=cfg.NDCG_K)
    bm25_recall = compute_recall(bm25_doc_ids, qrels_for_query, k=cfg.RECALL_K)
    hyde_recall = compute_recall(hyde_doc_ids, qrels_for_query, k=cfg.RECALL_K)

    records.append({
        "query_id": query_id,
        "query_text": row["query_text"],
        "pseudo_document": row["pseudo_document"],
        "bm25_ndcg": bm25_ndcg,
        "hyde_ndcg": hyde_ndcg,
        "ndcg_gain": hyde_ndcg - bm25_ndcg,
        "bm25_recall": bm25_recall,
        "hyde_recall": hyde_recall,
        "recall_gain": hyde_recall - bm25_recall,
        "bm25_latency_ms": row["bm25_latency_ms"],
        "hyde_latency_ms": row["hyde_latency_ms"],
        "pseudo_extra_tokens": row["pseudo_extra_tokens"],
    })

df = pd.DataFrame(records)
df.to_csv(cfg.OUTPUT_CSV, index=False)
print(f"Overwrote results with corrected qrels: {cfg.OUTPUT_CSV}")


# Cell 8: Summarize and save


In [8]:
df = pd.DataFrame(records)
df.to_csv(cfg.OUTPUT_CSV, index=False)

print(f"\nSaved per-query results to: {cfg.OUTPUT_CSV}")
print(f"Evaluated queries: {len(df)}")
print(f"Queries with qrels: {(df['bm25_ndcg'] + df['hyde_ndcg'] > 0).sum()}")

print("\n=== Overall Averages ===")
print(f"BM25 nDCG@{cfg.NDCG_K}:     {df['bm25_ndcg'].mean():.4f}")
print(f"HyDE nDCG@{cfg.NDCG_K}:     {df['hyde_ndcg'].mean():.4f}")
print(f"Mean nDCG gain:             {df['ndcg_gain'].mean():+.4f}")
print(f"Win rate (HyDE > BM25):     {(df['ndcg_gain'] > 0).mean():.1%}")

print(f"\nBM25 Recall@{cfg.RECALL_K}:   {df['bm25_recall'].mean():.4f}")
print(f"HyDE Recall@{cfg.RECALL_K}:   {df['hyde_recall'].mean():.4f}")
print(f"Mean Recall gain:           {df['recall_gain'].mean():+.4f}")

print(f"\nBM25 latency: {df['bm25_latency_ms'].mean():.1f} ms/query")
print(f"HyDE latency: {df['hyde_latency_ms'].mean():.1f} ms/query")

print("\n=== Top 10 nDCG gains ===")
print(df.sort_values("ndcg_gain", ascending=False)[[
    "query_id", "query_text", "ndcg_gain", "bm25_ndcg", "hyde_ndcg"
]].head(10).to_string(index=False))

print("\n=== Top 10 nDCG losses ===")
print(df.sort_values("ndcg_gain", ascending=True)[[
    "query_id", "query_text", "ndcg_gain", "bm25_ndcg", "hyde_ndcg"
]].head(10).to_string(index=False))



Saved per-query results to: c:\Users\hanaz\Documents\GitHub\Multi-Agent-Ensemble-for-Search-Through-Reinforcement-Optimization-MAESTRO-\outputs\hyde_isolation_results.csv
Evaluated queries: 20
Queries with qrels: 11

=== Overall Averages ===
BM25 nDCG@10:     0.3166
HyDE nDCG@10:     0.3004
Mean nDCG gain:             -0.0162
Win rate (HyDE > BM25):     15.0%

BM25 Recall@100:   0.5250
HyDE Recall@100:   0.5750
Mean Recall gain:           +0.0500

BM25 latency: 820.2 ms/query
HyDE latency: 74017.8 ms/query

=== Top 10 nDCG gains ===
query_id                                query_text  ndcg_gain  bm25_ndcg  hyde_ndcg
  524574                    trending topic meaning   0.698970   0.301030   1.000000
 1048779                         what is ott media   0.430677   0.000000   0.430677
 1048601                 what is pastoral medicine   0.200253   0.430677   0.630930
 1048673 what is ownership of a corporation called   0.000000   0.000000   0.000000
  944826            when do the oscar aw